# Generative Text Model - GPT Based Text Generation

## Task 4: Building a Coherent Text Generation Model

This notebook demonstrates a text generation model using GPT (Generative Pre-trained Transformer) that generates coherent paragraphs on specific topics based on user input prompts.

### Objective:
- Load a pre-trained GPT model
- Define functions to generate text from user prompts
- Accept user input and topics
- Generate and display high-quality coherent text
- Evaluate the quality of generated content

## Section 1: Import Required Libraries

Import necessary libraries including transformers, torch, and numpy for building and running the generative text model.

In [ ]:
# Import Required Libraries
import torch
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Section 2: Load Pre-trained GPT Model

Load a pre-trained GPT-2 model from the Hugging Face transformers library along with the tokenizer.

In [ ]:
# Load Pre-trained GPT-2 Model and Tokenizer
print("Loading GPT-2 model and tokenizer...")

# Initialize the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Initialize the model
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Move model to the appropriate device (GPU or CPU)
model.to(device)

# Set model to evaluation mode
model.eval()

print("✓ GPT-2 model loaded successfully!")
print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")

## Section 3: Define Text Generation Function

Create a function that takes user prompts and generates coherent text using the pre-trained model with configurable parameters.

In [ ]:
def generate_text(prompt, max_length=150, temperature=0.7, top_p=0.9, num_return_sequences=1):
    """
    Generate text based on a user prompt using the GPT-2 model.
    
    Parameters:
    -----------
    prompt : str
        The input prompt to start text generation
    max_length : int
        Maximum length of generated text (default: 150)
    temperature : float
        Controls randomness (0.0-1.0). Lower = more deterministic, Higher = more creative
    top_p : float
        Nucleus sampling parameter (0.0-1.0). Filters out less likely tokens
    num_return_sequences : int
        Number of different sequences to generate
    
    Returns:
    --------
    list : Generated text sequences
    """
    
    # Encode the input prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # Generate text with no gradient computation
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            num_return_sequences=num_return_sequences,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            top_k=50
        )
    
    # Decode the generated sequences
    generated_texts = []
    for output_id in output_ids:
        text = tokenizer.decode(output_id, skip_special_tokens=True)
        generated_texts.append(text)
    
    return generated_texts

print("✓ Text generation function defined successfully!")

## Section 4: Create User Input Interface

Implement an input mechanism to accept user prompts and topics for text generation.

In [ ]:
# Define example prompts for demonstration
user_prompts = [
    "Artificial intelligence is transforming",
    "The future of technology",
    "Climate change and renewable energy",
    "The benefits of reading books",
    "Space exploration and its importance"
]

# Display available prompts
print("=" * 70)
print("AVAILABLE EXAMPLE PROMPTS")
print("=" * 70)
for i, prompt in enumerate(user_prompts, 1):
    print(f"{i}. {prompt}")
print("=" * 70)

# Configuration parameters
generation_config = {
    'max_length': 120,
    'temperature': 0.8,
    'top_p': 0.95,
    'num_return_sequences': 1
}

print("\n✓ Generation Configuration:")
print(f"  - Max Length: {generation_config['max_length']}")
print(f"  - Temperature: {generation_config['temperature']}")
print(f"  - Top-p (Nucleus): {generation_config['top_p']}")
print(f"  - Number of Sequences: {generation_config['num_return_sequences']}")

## Section 5: Generate Text Output

Execute the text generation function with the user input and display the generated paragraphs.

In [ ]:
# Store generated outputs
generated_outputs = {}

print("\n" + "=" * 70)
print("GENERATING TEXT FOR EACH PROMPT")
print("=" * 70 + "\n")

# Generate text for each prompt
for idx, prompt in enumerate(user_prompts, 1):
    print(f"Generating text for Prompt #{idx}...")
    
    # Generate text
    generated = generate_text(
        prompt=prompt,
        max_length=generation_config['max_length'],
        temperature=generation_config['temperature'],
        top_p=generation_config['top_p'],
        num_return_sequences=generation_config['num_return_sequences']
    )
    
    # Store the output
    generated_outputs[prompt] = generated[0]
    
    print(f"✓ Complete!\n")

print("=" * 70)
print(f"✓ Successfully generated text for all {len(user_prompts)} prompts!")
print("=" * 70)

## Section 6: Display and Evaluate Generated Text

Present the generated output in a readable format and provide quality assessment.

In [ ]:
# Display generated outputs with formatting
print("\n" + "=" * 80)
print("GENERATED TEXT OUTPUT")
print("=" * 80 + "\n")

for idx, (prompt, generated_text) in enumerate(generated_outputs.items(), 1):
    print(f"EXAMPLE #{idx}")
    print("-" * 80)
    print(f"📝 USER PROMPT:\n   {prompt}")
    print(f"\n✨ GENERATED TEXT:\n   {generated_text}")
    print(f"\n📊 TEXT STATISTICS:")
    print(f"   - Word Count: {len(generated_text.split())}")
    print(f"   - Character Count: {len(generated_text)}")
    print(f"   - Sentences (estimated): {generated_text.count('.') + generated_text.count('!') + generated_text.count('?')}")
    print("\n" + "=" * 80 + "\n")

# Evaluation metrics
print("QUALITY EVALUATION METRICS")
print("-" * 80)

total_words = sum(len(text.split()) for text in generated_outputs.values())
total_chars = sum(len(text) for text in generated_outputs.values())
avg_words_per_generation = total_words / len(generated_outputs)

print(f"\n✓ Total Outputs Generated: {len(generated_outputs)}")
print(f"✓ Total Words Generated: {total_words}")
print(f"✓ Total Characters Generated: {total_chars}")
print(f"✓ Average Words per Generation: {avg_words_per_generation:.2f}")
print(f"✓ Model: GPT-2 (Pre-trained)")
print(f"✓ Device Used: {device}")
print(f"✓ Generation Parameters: Temperature={generation_config['temperature']}, Top-p={generation_config['top_p']}")

print("\n" + "=" * 80)
print("✓ TEXT GENERATION TASK COMPLETED SUCCESSFULLY!")
print("=" * 80)

## Advanced: Generate Text with Custom Parameters

Demonstrate advanced usage with different temperature and creativity settings.

In [ ]:
print("\n" + "=" * 80)
print("ADVANCED: COMPARING DIFFERENT TEMPERATURE SETTINGS")
print("=" * 80 + "\n")

# Test prompt
test_prompt = "Innovation in technology"

# Different temperature settings to demonstrate creativity levels
temperature_settings = {
    'Conservative (0.3)': 0.3,
    'Balanced (0.7)': 0.7,
    'Creative (0.9)': 0.9
}

print(f"Test Prompt: '{test_prompt}'\n")

for setting_name, temp_value in temperature_settings.items():
    print(f"\n{setting_name}")
    print("-" * 80)
    
    generated = generate_text(
        prompt=test_prompt,
        max_length=100,
        temperature=temp_value,
        top_p=0.9,
        num_return_sequences=1
    )
    
    print(f"Generated: {generated[0]}\n")

print("=" * 80)
print("NOTES ON TEMPERATURE:")
print("-" * 80)
print("• LOW (0.3): More deterministic, predictable, factual output")
print("• MEDIUM (0.7): Balanced creativity and coherence")
print("• HIGH (0.9): More creative, varied, potentially less predictable")
print("=" * 80)

## Summary

### Task Completed: Generative Text Model ✓

This Jupyter Notebook successfully demonstrates:

1. **Model Selection**: Used GPT-2 (pre-trained transformer model) for text generation
2. **Library Integration**: Leveraged Hugging Face `transformers` library for easy model access
3. **Text Generation Function**: Created a reusable function with configurable parameters:
   - `max_length`: Controls output length
   - `temperature`: Controls creativity (0.0-1.0)
   - `top_p`: Controls diversity through nucleus sampling
   - `num_return_sequences`: Generate multiple variations

4. **User Input Processing**: Demonstrated with multiple example prompts on diverse topics
5. **Generated Output**: Showcased coherent paragraph generation for each prompt
6. **Quality Metrics**: Provided word counts, character counts, and sentence analysis
7. **Advanced Features**: Demonstrated parameter tuning for different creativity levels

### Key Capabilities:
- ✓ Generates coherent multi-sentence paragraphs
- ✓ Supports diverse topics through flexible prompts
- ✓ Configurable generation parameters for different use cases
- ✓ GPU acceleration support (auto-detects CUDA availability)
- ✓ Reproducible results with seed setting

### Model Information:
- **Architecture**: GPT-2 (117M parameters)
- **Training Data**: WebText dataset (45GB of internet text)
- **Framework**: PyTorch via Hugging Face Transformers

### Further Improvements:
- Fine-tune model on domain-specific data for specialized applications
- Use larger models (GPT-3, GPT-4) for better quality (requires API)
- Implement prompt templates for specific genres or styles
- Add text quality filtering and ranking systems